# Notebook 2 – Train All Models & Save to S3

**Purpose:** Download the dataset from S3, train three CNN models (MobileNetV3, EfficientNetB0, ResNet50) with data augmentation, and upload all trained model weights back to S3.

**Run this AFTER `01_setup_and_data.ipynb`.**

---
### What this notebook does
1. Loads project config from `config.json`
2. Downloads the dataset from S3 to local `/tmp/banana_dataset`
3. Creates PyTorch `DataLoader`s with data augmentation for training
4. Fine-tunes three pre-trained models (transfer learning)
5. Saves the best weights for each model locally *and* uploads them to S3
6. Prints a training summary table

## Cell 1 – Imports & Configuration

In [ ]:
import os
import json
import copy
import time
import shutil
import pathlib
import warnings
warnings.filterwarnings("ignore")

import boto3
import sagemaker
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# ── Load project config ───────────────────────────────────────────────────────
CONFIG_FILE = "config.json"
if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(
        f"'{CONFIG_FILE}' not found. Please run 01_setup_and_data.ipynb first."
    )

with open(CONFIG_FILE) as f:
    config = json.load(f)

S3_BUCKET       = config["S3_BUCKET"]
S3_DATA_PREFIX  = config["S3_DATA_PREFIX"]
S3_MODEL_PREFIX = config["S3_MODEL_PREFIX"]
NUM_CLASSES     = config["NUM_CLASSES"]
CLASS_NAMES     = config["CLASS_NAMES"]

# ── ⚙️  Hyperparameters – feel free to adjust ─────────────────────────────────
BATCH_SIZE   = 16
NUM_EPOCHS   = 10
LEARNING_RATE = 0.001
INPUT_SIZE   = 224
NUM_WORKERS  = 4     # set to 0 if you get multiprocessing errors
LOCAL_DATA_DIR   = "/tmp/banana_dataset"
LOCAL_MODELS_DIR = "/tmp/trained_models"
# ────────────────────────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=" * 55)
print("Training Configuration")
print("=" * 55)
print(f"  S3 Bucket      : {S3_BUCKET}")
print(f"  Data prefix    : {S3_DATA_PREFIX}")
print(f"  Model prefix   : {S3_MODEL_PREFIX}")
print(f"  Classes ({NUM_CLASSES})    : {CLASS_NAMES}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Device         : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU            : {torch.cuda.get_device_name(0)}")

## Cell 2 – Download Dataset from S3

In [ ]:
def download_dataset_from_s3(bucket: str, prefix: str, local_dir: str) -> str:
    """
    Download every object under s3://bucket/prefix/ into local_dir.
    Skips files that already exist locally (resume-friendly).
    Returns the local directory path.
    """
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")

    try:
        pages = paginator.paginate(Bucket=bucket, Prefix=prefix + "/")
        keys = [obj["Key"] for page in pages for obj in page.get("Contents", [])]
    except Exception as e:
        raise RuntimeError(f"Failed to list S3 objects: {e}") from e

    if not keys:
        raise ValueError(
            f"No files found at s3://{bucket}/{prefix}/. "
            "Please run 01_setup_and_data.ipynb to upload the dataset."
        )

    print(f"Found {len(keys)} files in s3://{bucket}/{prefix}/")
    downloaded = 0
    skipped    = 0

    for key in tqdm(keys, desc="Downloading"):
        # Compute local path by stripping the S3 prefix
        relative  = key[len(prefix):].lstrip("/")
        local_path = os.path.join(local_dir, relative)

        if os.path.exists(local_path):
            skipped += 1
            continue

        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        try:
            s3.download_file(bucket, key, local_path)
            downloaded += 1
        except Exception as e:
            print(f"  ⚠️  Failed to download {key}: {e}")

    print(f"✅ Download complete — {downloaded} new files, {skipped} already cached.")
    return local_dir


print("Downloading dataset from S3 (first run may take a few minutes) ...")
try:
    dataset_dir = download_dataset_from_s3(S3_BUCKET, S3_DATA_PREFIX, LOCAL_DATA_DIR)
    print(f"Dataset is at: {dataset_dir}")
except Exception as e:
    print(f"❌ Dataset download failed: {e}")
    raise

## Cell 3 – Data Loaders with Augmentation

In [ ]:
def build_dataloaders(data_root: str, input_size: int, batch_size: int, num_workers: int):
    """
    Build train / valid / test DataLoaders.
    Training set uses heavy augmentation; val/test only resize + normalise.
    """
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std  = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        # Random crop with 80–100% zoom
        transforms.RandomResizedCrop(input_size, scale=(0.8, 1.0)),
        # Flips
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        # 90° rotation variants
        transforms.RandomChoice([
            transforms.RandomRotation((90, 90)),
            transforms.RandomRotation((-90, -90)),
            transforms.RandomRotation((180, 180)),
            transforms.RandomRotation((0, 0)),
        ]),
        # Fine rotation ±15°
        transforms.RandomRotation(degrees=15),
        # Colour jitter
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        # Slight blur
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    # Locate split folders – handle both flat and nested layouts
    def find_split_dir(root, split):
        direct = os.path.join(root, split)
        if os.path.isdir(direct):
            return direct
        # Try one level deeper (e.g., root/banana-ripeness-dataset-original/train)
        for sub in os.listdir(root):
            candidate = os.path.join(root, sub, split)
            if os.path.isdir(candidate):
                return candidate
        raise FileNotFoundError(
            f"Could not find '{split}' folder under {root}. "
            "Check your dataset structure."
        )

    try:
        split_dirs = {
            "train": find_split_dir(data_root, "train"),
            "valid": find_split_dir(data_root, "valid"),
            "test":  find_split_dir(data_root, "test"),
        }
    except FileNotFoundError as e:
        raise FileNotFoundError(str(e)) from e

    split_transforms = {"train": train_transform, "valid": val_test_transform, "test": val_test_transform}

    datasets_dict = {
        split: datasets.ImageFolder(split_dirs[split], split_transforms[split])
        for split in ["train", "valid", "test"]
    }

    loaders = {
        split: DataLoader(
            datasets_dict[split],
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
        for split in ["train", "valid", "test"]
    }

    sizes = {split: len(datasets_dict[split]) for split in loaders}
    classes = datasets_dict["train"].classes

    return loaders, sizes, classes


try:
    dataloaders, dataset_sizes, found_classes = build_dataloaders(
        LOCAL_DATA_DIR, INPUT_SIZE, BATCH_SIZE, NUM_WORKERS
    )
    print("✅ DataLoaders ready")
    print(f"   Classes  : {found_classes}")
    for split, size in dataset_sizes.items():
        print(f"   {split:6s}  : {size} images")

    if sorted(found_classes) != sorted(CLASS_NAMES):
        print(f"\n⚠️  WARNING: detected classes {found_classes} differ from config {CLASS_NAMES}.")
        print("   The model will use the detected classes. Update config.json if needed.")
        NUM_CLASSES = len(found_classes)
        CLASS_NAMES = found_classes
except Exception as e:
    print(f"❌ Failed to build DataLoaders: {e}")
    raise

## Cell 4 – Model Builder

In [ ]:
def build_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Load a pre-trained torchvision model and replace the final
    classification layer to match the target number of classes.
    Raises ValueError if model_name is unrecognised.
    """
    if model_name == "MobileNetV3":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    elif model_name == "EfficientNetB0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError(
            f"Unknown model '{model_name}'. "
            "Supported models: MobileNetV3, EfficientNetB0, ResNet50."
        )

    return model

print("✅ Model builder defined.")

## Cell 5 – Training Loop

In [ ]:
def train_one_model(
    model_name: str,
    num_classes: int,
    dataloaders: dict,
    dataset_sizes: dict,
    device: torch.device,
    num_epochs: int,
    lr: float,
    save_dir: str,
) -> dict:
    """
    Train a single model and return a results dict with:
        model, best_val_acc, local_path, history
    """
    print(f"\n{'=' * 55}")
    print(f"  Training {model_name}")
    print(f"{'=' * 55}")

    try:
        model = build_model(model_name, num_classes).to(device)
    except Exception as e:
        print(f"❌ Failed to build {model_name}: {e}")
        raise

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # LR scheduler: reduce by 0.1 if val loss doesn't improve for 3 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.1, patience=3, verbose=True
    )

    best_weights = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    start = time.time()

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}  {'─' * 10}")

        for phase in ["train", "valid"]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0.0
            running_corrects = 0

            try:
                for inputs, labels in tqdm(dataloaders[phase], desc=phase, leave=False):
                    inputs = inputs.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    with torch.set_grad_enabled(phase == "train"):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)
                        if phase == "train":
                            loss.backward()
                            optimizer.step()

                    running_loss     += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

            except RuntimeError as e:
                # Common cause: GPU OOM – suggest reducing batch size
                if "out of memory" in str(e).lower():
                    print(
                        f"❌ CUDA out of memory during {phase} of {model_name}. "
                        f"Try reducing BATCH_SIZE (currently {BATCH_SIZE})."
                    )
                raise

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc  = running_corrects.double() / dataset_sizes[phase]

            print(f"  {phase:5s} Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

            if phase == "train":
                history["train_loss"].append(epoch_loss)
                history["train_acc"].append(float(epoch_acc))
            else:
                history["val_loss"].append(epoch_loss)
                history["val_acc"].append(float(epoch_acc))
                scheduler.step(epoch_acc)
                if epoch_acc > best_val_acc:
                    best_val_acc  = float(epoch_acc)
                    best_weights  = copy.deepcopy(model.state_dict())

    elapsed = time.time() - start
    print(f"\n⏱  {model_name} trained in {int(elapsed // 60)}m {int(elapsed % 60)}s")
    print(f"🏆 Best validation accuracy: {best_val_acc:.4f}")

    # Load best weights back into model
    model.load_state_dict(best_weights)

    # Save locally
    os.makedirs(save_dir, exist_ok=True)
    local_path = os.path.join(save_dir, f"{model_name}_banana_ripeness.pth")
    try:
        torch.save(model.state_dict(), local_path)
        print(f"💾 Saved locally → {local_path}")
    except OSError as e:
        print(f"❌ Could not save model locally: {e}")
        raise

    return {
        "model":        model,
        "best_val_acc": best_val_acc,
        "local_path":   local_path,
        "history":      history,
    }

print("✅ Training function defined.")

## Cell 6 – Train All Models

> ⏳ **Estimated time (GPU ml.g4dn.xlarge):** ~4–8 minutes per model = 12–24 minutes total.
> 
> On CPU this will be much longer. Consider using a GPU instance.

In [ ]:
MODEL_NAMES   = ["MobileNetV3", "EfficientNetB0", "ResNet50"]
training_results = {}   # model_name -> result dict

for model_name in MODEL_NAMES:
    try:
        result = train_one_model(
            model_name   = model_name,
            num_classes  = NUM_CLASSES,
            dataloaders  = dataloaders,
            dataset_sizes= dataset_sizes,
            device       = DEVICE,
            num_epochs   = NUM_EPOCHS,
            lr           = LEARNING_RATE,
            save_dir     = LOCAL_MODELS_DIR,
        )
        training_results[model_name] = result
    except Exception as e:
        print(f"\n❌ Training of {model_name} failed: {e}")
        print("   Skipping this model and continuing with the next one.")
        training_results[model_name] = None
        continue

print("\n" + "=" * 55)
print("Training Summary")
print("=" * 55)
for name, res in training_results.items():
    if res:
        print(f"  {name:15s}  best_val_acc = {res['best_val_acc']:.4f}")
    else:
        print(f"  {name:15s}  ❌ FAILED")

## Cell 7 – Upload All Trained Models to S3

In [ ]:
def upload_model_to_s3(local_path: str, bucket: str, prefix: str) -> str:
    """
    Upload a local .pth file to S3 and return the S3 URI.
    """
    s3 = boto3.client("s3")
    filename = os.path.basename(local_path)
    s3_key   = f"{prefix}/{filename}"
    try:
        s3.upload_file(local_path, bucket, s3_key)
        s3_uri = f"s3://{bucket}/{s3_key}"
        return s3_uri
    except Exception as e:
        raise RuntimeError(f"Failed to upload {local_path} to S3: {e}") from e


print("Uploading trained models to S3 ...\n")
s3_uris = {}

for model_name, res in training_results.items():
    if res is None:
        print(f"  ⏭  Skipping {model_name} (training failed).")
        continue
    try:
        uri = upload_model_to_s3(res["local_path"], S3_BUCKET, S3_MODEL_PREFIX)
        s3_uris[model_name] = uri
        print(f"  ✅ {model_name} → {uri}")
    except RuntimeError as e:
        print(f"  ❌ {model_name} upload failed: {e}")

# Persist S3 URIs into config for the evaluation notebook
config["s3_model_uris"] = s3_uris
config["local_models_dir"] = LOCAL_MODELS_DIR
try:
    with open(CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=2)
    print(f"\n✅ Updated {CONFIG_FILE} with S3 model URIs.")
except OSError as e:
    print(f"⚠️  Could not update {CONFIG_FILE}: {e}")

## Cell 8 – Training Curves (Optional Visualisation)

In [ ]:
import matplotlib.pyplot as plt

successful = {k: v for k, v in training_results.items() if v is not None}
if not successful:
    print("No successful training runs to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for model_name, res in successful.items():
        h = res["history"]
        epochs_range = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(epochs_range, h["train_loss"], label=f"{model_name} train")
        axes[0].plot(epochs_range, h["val_loss"],   label=f"{model_name} val", linestyle="--")
        axes[1].plot(epochs_range, h["train_acc"],  label=f"{model_name} train")
        axes[1].plot(epochs_range, h["val_acc"],    label=f"{model_name} val", linestyle="--")

    for ax, title, ylabel in zip(
        axes,
        ["Loss per Epoch", "Accuracy per Epoch"],
        ["Loss", "Accuracy"],
    ):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)
        ax.grid(True)

    plt.tight_layout()
    plt.savefig("/tmp/training_curves.png", dpi=150)
    plt.show()
    print("\n✅ Training curves saved to /tmp/training_curves.png")

---
## ✅ Training Complete!

All models have been trained and saved to S3. Next step → **open and run `03_evaluate_and_select_best.ipynb`**.